# Beyond the US: The Data Job Market Excluding US Postings

**Goal:** Check whether the trends found in previous notebooks still hold when US-based postings are excluded, since the dataset skews heavily toward the US.

**Input:** `data_jobs_clean.parquet`, `data_jobs_salary.parquet`

**What this notebook covers:**
- Top skills in demand, excluding the US
- Salary patterns, excluding the US
- Comparison with the global (US-included) findings
- Key visualizations and takeaways

---

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

pd.options.display.float_format = '{:,.0f}'.format

def currency_formatter(x, pos):
    return f'${x/1000:,.0f}K'

def count_formatter(x, pos):
    return f'{x/1000:,.0f}K' if x >= 1000 else f'{x:.0f}'

df = pd.read_parquet('../data/clean/data_jobs_clean.parquet')
df_salary = pd.read_parquet('../data/clean/data_jobs_salary.parquet')

df_non_us = df[df['job_country'] != 'United States'].copy()
df_salary_non_us = df_salary[df_salary['job_country'] != 'United States'].copy()

print(f"Non-US postings: {len(df_non_us):,} ({len(df_non_us)/len(df)*100:.1f}% of total)")
print(f"Non-US salary postings: {len(df_salary_non_us):,} ({len(df_salary_non_us)/len(df_salary)*100:.1f}% of total)")

Non-US postings: 579,449 (73.7% of total)
Non-US salary postings: 7,494 (22.9% of total)


**Finding**: Non-US postings make up 73.7% of the dataset by volume, showing this isn't purely a US-only dataset despite the earlier salary bias. However, only 22.9% of salary-labeled postings are non-US, meaning US employers disclose salary information far more often, proportionally, than non-US employers. This means salary-based conclusions below should be read with a note of caution: the non-US salary sample, while usable, is proportionally thinner than its share of overall job volume.

## Top Skills in Demand, Excluding the US

In [2]:
df_non_us_skills = df_non_us[df_non_us['job_skills'].notna()].copy()
df_non_us_exploded = df_non_us_skills.explode('job_skills')

top_skills_non_us = (
    df_non_us_exploded['job_skills']
    .value_counts()
    .head(15)
)
top_skills_non_us

job_skills
python        273733
sql           269801
aws           107231
azure         104744
spark          84431
excel          81620
r              79639
tableau        77617
power bi       71588
java           62752
hadoop         46138
sas            46040
gcp            41640
scala          41365
databricks     40124
Name: count, dtype: int64

**Finding**: The top in-demand skills outside the US closely mirror the global ranking, with Python and SQL still dominating by a wide margin. This suggests the core skill demand trends found earlier aren't simply an artifact of US market dominance. One notable difference: Hadoop and Scala appear in this non-US top 15 (replacing Snowflake and Airflow from the global ranking), hinting at a slightly stronger presence of traditional big data stacks outside the US.

## Salary Patterns, Excluding the US

In [3]:
salary_comparison_non_us = (
    df_salary_non_us.groupby('job_work_from_home')['salary_year_avg']
    .agg(['median', 'count'])
    .rename(index={False: 'On-site', True: 'Remote'})
)
salary_comparison_non_us

,median,count
job_work_from_home,,
On-site,"108,900",5575
Remote,"131,064",662


**Finding**: The remote salary premium is even more pronounced outside the US: +20.4% (vs +12% globally). This is driven mainly by a lower on-site median outside the US ($108,900 vs $115,000 globally), while the remote median stays close to the global figure ($131,064 vs $128,830). In other words, remote work appears to let non-US candidates access salary levels much closer to global (often US-influenced) standards, rather than being capped by local on-site pay.

**Caveat**: this is based on 662 non-US remote postings with salary data, a smaller sample than the global comparison, so this figure should be read as a strong directional signal rather than a precise estimate.

## Summary

- Non-US postings represent 73.7% of the dataset by volume, showing this isn't a purely US-centric dataset despite the earlier salary reporting bias. However, only 22.9% of salary-labeled postings are non-US, meaning US employers disclose